In [2]:
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
from datetime import datetime, timedelta

sheet_id = "1VIJzTfObi9ALe9dSS8Y6soxjlFIR5VNr68u_wrKrwMU"
sheet_name = "Sheet1"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
groups_df = pd.read_csv(url)
df = groups_df[["group_name","headcount","start","end","bld","contact", "start_a", "end_a"]]
# df['start_a'] = pd.to_datetime(df['start_a']).dt.date
# df['end_a'] = pd.to_datetime(df['end_a']).dt.date

from datetime import datetime, timedelta
import re

def expand_bld(start, end, bld, default_headcount=None):
    # if start and end are strings like "2025-10-18"
    start_date = datetime.strptime(start, "%m/%d/%Y")
    # start_date = datetime.strptime(start, "%m/%d/%Y")
    end_date = datetime.strptime(end, "%m/%d/%Y")
    # end_date = datetime.strptime(end, "%m/%d/%Y")
    start_str = start_date.strftime("%m/%d/%Y")
    end_str = end_date.strftime("%m/%d/%Y")

    bld_str = str(bld)
    
    day_meals = [x.strip().upper() for x in bld_str.split(",")]
    num_days = (end_date - start_date).days + 1
    dates = [start_date + timedelta(days=i) for i in range(num_days)]
    
    expanded = []
    
    prefixed = {
        "OB": "Outdoor Breakfast",
        "OL": "Outdoor Lunch",
        "OD": "Outdoor Dinner",
        "UB": "Upgraded Breakfast",
        "UL": "Upgraded Lunch",
        "UD": "Upgraded Dinner"
    }
    meal_map = {"B": "Breakfast", "L": "Lunch", "D": "Dinner"}

    for date, code in zip(dates, day_meals):
        meals, headcounts = [], []

        # Extract all meal variations including prefixed ones
        # e.g., BL(30)D(25)UB(10) -> { 'L': 30, 'D': 25, 'UB': 10 }
        variations = {m.group(1): int(m.group(2))
                      for m in re.finditer(r'([A-Z]{1,2})\((\d+)\)', code)}

        # Remove parentheses for easier pattern matching
        code = re.sub(r'\(\d+\)', '', code)

        # Handle prefixed meals first
        for key, label in prefixed.items():
            if key in code:
                meals.append(label)
                headcounts.append(variations.get(key, default_headcount))

        # Handle base meals (skip if prefix already applied)
        for key, label in meal_map.items():
            if key in code and not any(x in code for x in [f"O{key}", f"U{key}"]):
                meals.append(label)
                headcounts.append(variations.get(key, default_headcount))

        expanded.append({
            "date": date.strftime("%m/%d/%Y"),
            "meals": meals,
            "headcounts": headcounts
        })

    return expanded


# These are the permissions your script will request.
# Since your script creates a Form, it needs the 'forms' scope.
SCOPES = [
    "https://www.googleapis.com/auth/forms",
    "https://www.googleapis.com/auth/drive" # Needed if using the template method
]


# Replace with the Script ID from your deployment
SCRIPT_ID = "AKfycbyuisjIbd8pzkWHA9I14FuC4ug0C7idvg222LwdxHAkiqi10yUcyqBHr-4Gdr2uMGLg"

def main():
    """Calls the Apps Script API to run our form creation function."""
    creds = None
    # The file token.json stores the user's access and refresh tokens.
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)

    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("desktop_credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    service = build("script", "v1", credentials=creds)

    # Define the parameters for your Apps Script function
    for id,row in df.iterrows():
        form_name = row["group_name"] + " (" + datetime.strptime(row['start'], "%Y-%m-%d").strftime("%b %d, %Y") +") " + "Dietary Form"
        start_date = datetime.strptime(row['start'], "%Y-%m-%d").strftime("%m/%d/%Y")
        end_date = datetime.strptime(row['end'], "%Y-%m-%d").strftime("%m/%d/%Y")
        headcount = row["headcount"]
        bld = row["bld"]
        bld_data = expand_bld(start_date, end_date, bld, default_headcount=headcount)
        

        # Create the request body
        request = {
            "function": "create_dietary_form",
            "parameters": [form_name, start_date, end_date, headcount, bld_data],
            "devMode": True # Use devMode to run the latest saved code
        }

        # Make the API call
        print("Calling the Apps Script API...")
        response = service.scripts().run(scriptId=SCRIPT_ID, body=request).execute()
        print(response)
        
        

        if "error" in response:
            error = response["error"]["details"][0]
            print(f"Script error message: {error['errorMessage']}")
            print(f"Script error stacktrace: {error['scriptStackTraceElements']}")
        else:
            # The response.get('response').get('result') will contain what your
            # Apps Script function returns. Since ours just logs a URL, this will be null.
            print("Function executed successfully!")

if __name__ == "__main__":
    main()

Calling the Apps Script API...
{'done': True, 'response': {'@type': 'type.googleapis.com/google.apps.script.v1.ExecutionResponse', 'result': {'editUrl': 'https://docs.google.com/forms/d/1IKnXKpeWDqvtwoX47VBD8FSqwujF2iZdN2YVwGFdfew/edit', 'publishedUrl': 'https://docs.google.com/forms/d/e/1FAIpQLSfrW7L66SCHL7K0hLIe7QIsv89K6jssq2yf9ggTuBJDLUTtrA/viewform'}}}
Function executed successfully!
Calling the Apps Script API...
{'done': True, 'response': {'@type': 'type.googleapis.com/google.apps.script.v1.ExecutionResponse', 'result': {'editUrl': 'https://docs.google.com/forms/d/184NI3oYuPek7kmckQxGphoV4pUEoNyieDm6TUEW4eXM/edit', 'publishedUrl': 'https://docs.google.com/forms/d/e/1FAIpQLScQmgd2aADTJnvDrJC4zB_5XgNN9FX5UPuoMeb8xsA_2370ZQ/viewform'}}}
Function executed successfully!
Calling the Apps Script API...
{'done': True, 'response': {'@type': 'type.googleapis.com/google.apps.script.v1.ExecutionResponse', 'result': {'editUrl': 'https://docs.google.com/forms/d/1bpt1E2WuTdTon9zADTAj2pKTgBcULsJ

In [26]:
from datetime import datetime, timedelta
import re

def expand_bld(start, end, bld, default_headcount=None):
    start_date = datetime.strptime(start, "%m/%d/%Y")
    end_date = datetime.strptime(end, "%m/%d/%Y")
    bld_str = str(bld)
    
    day_meals = [x.strip().upper() for x in bld_str.split(",")]
    num_days = (end_date - start_date).days + 1
    dates = [start_date + timedelta(days=i) for i in range(num_days)]
    
    expanded = []
    
    prefixed = {
        "OB": "Outdoor Breakfast",
        "OL": "Outdoor Lunch",
        "OD": "Outdoor Dinner",
        "UB": "Upgraded Breakfast",
        "UL": "Upgraded Lunch",
        "UD": "Upgraded Dinner"
    }
    meal_map = {"B": "Breakfast", "L": "Lunch", "D": "Dinner"}

    for date, code in zip(dates, day_meals):
        meals, headcounts = [], []

        # Extract all meal variations including prefixed ones
        # e.g., BL(30)D(25)UB(10) -> { 'L': 30, 'D': 25, 'UB': 10 }
        variations = {m.group(1): int(m.group(2))
                      for m in re.finditer(r'([A-Z]{1,2})\((\d+)\)', code)}

        # Remove parentheses for easier pattern matching
        code = re.sub(r'\(\d+\)', '', code)

        # Handle prefixed meals first
        for key, label in prefixed.items():
            if key in code:
                meals.append(label)
                headcounts.append(variations.get(key, default_headcount))

        # Handle base meals (skip if prefix already applied)
        for key, label in meal_map.items():
            if key in code and not any(x in code for x in [f"O{key}", f"U{key}"]):
                meals.append(label)
                headcounts.append(variations.get(key, default_headcount))

        expanded.append({
            "date": date.strftime("%m/%d/%Y"),
            "meals": meals,
            "headcounts": headcounts
        })

    return expanded


In [3]:
import pandas as pd
sheet_id = "1FMYzd0Rs7RM6ylW34xy4sKz1vLz_h_PkAGhxsnoi82M"
sheet_name = "Sheet1"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"
groups_df = pd.read_csv(url)
groups_df = groups_df[["group_name","headcount","start","end","bld","contact"]]
groups_df

HTTPError: HTTP Error 401: Unauthorized

In [45]:
from datetime import datetime, timedelta
import re

def expand_bld(start, end, bld, default_headcount=None, default_times=None):
    start_date = datetime.strptime(start, "%m/%d/%Y")
    end_date = datetime.strptime(end, "%m/%d/%Y")
    bld_str = str(bld)
    
    day_meals = [x.strip().upper() for x in bld_str.split(",")]
    num_days = (end_date - start_date).days + 1
    dates = [start_date + timedelta(days=i) for i in range(num_days)]
    
    expanded = []
    
    prefixed = {
        "OB": "Outdoor Breakfast",
        "OL": "Outdoor Lunch",
        "OD": "Outdoor Dinner",
        "UB": "Upgraded Breakfast",
        "UL": "Upgraded Lunch",
        "UD": "Upgraded Dinner"
    }
    meal_map = {"B": "Breakfast", "L": "Lunch", "D": "Dinner"}
    meal_order = ["Breakfast", "Lunch", "Dinner"]

    # Default times if none specified
    meal_times_map = {
        "Breakfast": "8:00 AM",
        "Lunch": "12:00 PM",
        "Dinner": "5:00 PM"
    }
    if default_times:
        meal_times_map.update(default_times)

    for date, code in zip(dates, day_meals):
        meals, headcounts, meal_times = [], [], []

        # Extract variations like D(30@6:30), UB(@7:15)
        # Groups: (1)=meal, (2)=headcount, (3)=time
        pattern = r'([A-Z]{1,2})\((?:(\d+)?@?([\d:AMPamp\s]+)?)\)'
        variations = {
            m.group(1): {
                "headcount": int(m.group(2)) if m.group(2) else default_headcount,
                "time": m.group(3).strip() if m.group(3) else None
            }
            for m in re.finditer(pattern, code)
        }

        # Clean up meal codes by removing parentheses
        code = re.sub(r'\([^)]*\)', '', code)

        temp = []

        # Prefixed meals (UB, OL, etc.)
        for key, label in prefixed.items():
            if key in code:
                v = variations.get(key, {})
                headcount = v.get("headcount", default_headcount)
                time = v.get("time") or meal_times_map.get(label.split()[-1])
                temp.append((label, headcount, time))

        # Base meals (B, L, D) — skip if prefixed already exists
        for key, label in meal_map.items():
            if key in code and not any(x in code for x in [f"O{key}", f"U{key}"]):
                v = variations.get(key, {})
                headcount = v.get("headcount", default_headcount)
                time = v.get("time") or meal_times_map.get(label)
                temp.append((label, headcount, time))

        # Sort Breakfast → Lunch → Dinner
        temp.sort(key=lambda x: meal_order.index(x[0].split()[-1]))

        # Extract final lists
        for meal, count, time in temp:
            meals.append(meal)
            headcounts.append(count)
            meal_times.append(time)

        expanded.append({
            "date": date.strftime("%m/%d/%Y"),
            "meals": meals,
            "headcounts": headcounts,
            "meal_times": meal_times
        })

    return expanded


In [46]:
expand_bld(
    "10/15/2025", "10/17/2025",
    "D(80), B(2@11:00 PM)L(30)D, L",
    default_headcount=20
)


[{'date': '10/15/2025',
  'meals': ['Dinner'],
  'headcounts': [80],
  'meal_times': ['5:00 PM']},
 {'date': '10/16/2025',
  'meals': ['Breakfast', 'Lunch', 'Dinner'],
  'headcounts': [2, 30, 20],
  'meal_times': ['11:00 PM', '12:00 PM', '5:00 PM']},
 {'date': '10/17/2025',
  'meals': ['Lunch'],
  'headcounts': [20],
  'meal_times': ['12:00 PM']}]

In [ ]:
#what do I need?
#form_name -> group_name(start - end)
#start_date -> start (yyy-mm-dd)
#end_date -> end (yyy-mm-dd)
#headcount 
#bld -> expanded bld data structure

In [47]:
groups_df

,group_name,headcount,start,end,bld,contact
0,John Carroll,56,11/03/2025,11/04/2025,"D, BL",Rhys Gadbois: rgadbois@johncarroll.org\r\nKen ...
1,Good Counsel Juniors,60,11/5/2025,11/7/2025,"LD, BLD, BL",Maggie Bykowski - mbykowski@olgchs.org\r
2,Sr. Helen Memorial Retreat,24,11/7/2025,11/9/2025,"UD, BULUD, B",Carol O'Hara\r\noharacarol07@gmail.com\r\n443-...
3,Cristo Rey Sophomore,100,11/10/2025,11/10/2025,L,Larry Parr- lparr@cristoreybalt.org
4,Loyola Neighbors,30,11/12/2025,11/14/2025,"D, BLD. BL",Bernie Bowers- BBowers@LoyolaBlakefield.org
5,Board/Staff Retreat,25,11/15/2025,11/15/2025,LD,odwyerretreathouse@gmail.com
6,St. Ignatius Hickory,60,11/15/2025,11/15/2025,NaN,Caleb Olsen: caleb.olsen@archbalt.org
7,John Carroll,56,11/17/2025,11/18/2025,"D, BL",Rhys Gadbois: rgadbois@johncarroll.org\r\nKen ...
8,Spalding,61,11/19/2025,11/21/2025,"D, BLD, BL",Caitlin Boyle: boylec@spaldinghs.org
9,MSJ,42,11/23/2025,11/25/2025,"D, BLD, BL",Clay Bonham- cbonham@msjnet.edu


In [53]:
result = {
  "form_name":None,
  "start_date":None,
  "end_date":None,
  "headcount":None,
  "bld": None
}
df = groups_df
df
  

,group_name,headcount,start,end,bld,contact
0,John Carroll,56,11/03/2025,11/04/2025,"D, BL",Rhys Gadbois: rgadbois@johncarroll.org\r\nKen ...
1,Good Counsel Juniors,60,11/5/2025,11/7/2025,"LD, BLD, BL",Maggie Bykowski - mbykowski@olgchs.org\r
2,Sr. Helen Memorial Retreat,24,11/7/2025,11/9/2025,"UD, BULUD, B",Carol O'Hara\r\noharacarol07@gmail.com\r\n443-...
3,Cristo Rey Sophomore,100,11/10/2025,11/10/2025,L,Larry Parr- lparr@cristoreybalt.org
4,Loyola Neighbors,30,11/12/2025,11/14/2025,"D, BLD. BL",Bernie Bowers- BBowers@LoyolaBlakefield.org
5,Board/Staff Retreat,25,11/15/2025,11/15/2025,LD,odwyerretreathouse@gmail.com
6,St. Ignatius Hickory,60,11/15/2025,11/15/2025,NaN,Caleb Olsen: caleb.olsen@archbalt.org
7,John Carroll,56,11/17/2025,11/18/2025,"D, BL",Rhys Gadbois: rgadbois@johncarroll.org\r\nKen ...
8,Spalding,61,11/19/2025,11/21/2025,"D, BLD, BL",Caitlin Boyle: boylec@spaldinghs.org
9,MSJ,42,11/23/2025,11/25/2025,"D, BLD, BL",Clay Bonham- cbonham@msjnet.edu


In [54]:
for idx, group in df.iterrows():
  result["form_name"] = f"{group['group_name']} ({group['start']} - {group['end']})"
  result["start_date"] = datetime.strptime(group['start'], "%m/%d/%Y").strftime("%Y-%m-%d")
  result["end_date"] = datetime.strptime(group['end'], "%m/%d/%Y").strftime("%Y-%m-%d")
  result["headcount"] = group['headcount']
  result["bld"] = expand_bld(
      group['start'], group['end'], group['bld'],
      default_headcount=group['headcount']
  )
  print(result)

{'form_name': 'John Carroll (11/03/2025 - 11/04/2025)', 'start_date': '2025-11-03', 'end_date': '2025-11-04', 'headcount': 56, 'bld': [{'date': '11/03/2025', 'meals': ['Dinner'], 'headcounts': [56], 'meal_times': ['5:00 PM']}, {'date': '11/04/2025', 'meals': ['Breakfast', 'Lunch'], 'headcounts': [56, 56], 'meal_times': ['8:00 AM', '12:00 PM']}]}
{'form_name': 'Good Counsel Juniors (11/5/2025 - 11/7/2025)', 'start_date': '2025-11-05', 'end_date': '2025-11-07', 'headcount': 60, 'bld': [{'date': '11/05/2025', 'meals': ['Lunch', 'Dinner'], 'headcounts': [60, 60], 'meal_times': ['12:00 PM', '5:00 PM']}, {'date': '11/06/2025', 'meals': ['Breakfast', 'Lunch', 'Dinner'], 'headcounts': [60, 60, 60], 'meal_times': ['8:00 AM', '12:00 PM', '5:00 PM']}, {'date': '11/07/2025', 'meals': ['Breakfast', 'Lunch'], 'headcounts': [60, 60], 'meal_times': ['8:00 AM', '12:00 PM']}]}
{'form_name': 'Sr. Helen Memorial Retreat (11/7/2025 - 11/9/2025)', 'start_date': '2025-11-07', 'end_date': '2025-11-09', 'headc

In [ ]:
def make_dietary_form():
  #1 get data from sheets
  #2 for each row, expand bld(have function), create results dict and use those values to create form(have function) & invoice(have functuion)